In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
print("version",tf.__version__)
print("gpu available",tf.config.list_physical_devices("GPU"))

version 2.20.0
gpu available [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
Train_Dir = "/kaggle/input/datasets/anonngc/fer2013/train"
Test_Dir = "/kaggle/input/datasets/anonngc/fer2013/test"

In [3]:
IMG_SIZE = (224,224)
BATCH_size= 64

In [4]:
train_ds = keras.utils.image_dataset_from_directory(
    Train_Dir,image_size = IMG_SIZE,batch_size = BATCH_size,
    shuffle = True,label_mode='int',color_mode ='rgb')
test_ds = keras.utils.image_dataset_from_directory(
    Test_Dir,image_size= IMG_SIZE,
    batch_size= BATCH_size,
    shuffle= True,
    label_mode = 'int',
    color_mode = 'rgb'
)

Found 28709 files belonging to 7 classes.


I0000 00:00:1788179553.698412    1320 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 7178 files belonging to 7 classes.


In [5]:
class_name = train_ds.class_names
n_class = len(class_name)

print(f"all classes : \n{class_name}\ntotal no of classes:{n_class}")

all classes : 
['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
total no of classes:7


In [6]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [7]:
train_ds

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [8]:
base_model = keras.applications.VGG16(
    weights = "imagenet",
    include_top = False ,
    input_shape = (224,224,3)
)

In [9]:
print("VGG model loaded,final cnn layer output shape",base_model.output_shape)

VGG model loaded,final cnn layer output shape (None, 7, 7, 512)


In [13]:
base_model.trainable = False
input = keras.Input(shape=(224,224,3))
x = keras.applications.vgg16.preprocess_input(input)
x = base_model(x,training = False)
x = keras.layers.GlobalAveragePooling2D()(x)#flatten
output = keras.layers.Dense(n_class,activation = "softmax")(x)#final output

model = keras.Model(input,output)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ input_layer_4[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 7, 7, 512) │ 14,714,688 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[1][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 7)         │      3,591 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,718,279 (56.15 MB)

 Trainable params: 3,591 (14.03 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [15]:
model.compile(
    optimizer = keras.optimizers.Adam(learning_rate = 0.001),
    loss = "sparse_categorical_crossentropy",
    metrics = ['accuracy']
)

In [16]:
history = model.fit(train_ds,validation_data=test_ds,epochs=5)

Epoch 1/5
  1/449 ━━━━━━━━━━━━━━━━━━━━ 2:04:52 17s/step - accuracy: 0.1250 - loss: 5.4130

I0000 00:00:1788180337.327035    1387 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


449/449 ━━━━━━━━━━━━━━━━━━━━ 93s 171ms/step - accuracy: 0.3530 - loss: 1.9613 - val_accuracy: 0.4284 - val_loss: 1.6041
Epoch 2/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 65s 145ms/step - accuracy: 0.4545 - loss: 1.4875 - val_accuracy: 0.4695 - val_loss: 1.4564
Epoch 3/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 65s 145ms/step - accuracy: 0.4880 - loss: 1.3883 - val_accuracy: 0.4765 - val_loss: 1.4182
Epoch 4/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 65s 145ms/step - accuracy: 0.5010 - loss: 1.3435 - val_accuracy: 0.4861 - val_loss: 1.3955
Epoch 5/5
449/449 ━━━━━━━━━━━━━━━━━━━━ 65s 145ms/step - accuracy: 0.5066 - loss: 1.3240 - val_accuracy: 0.4901 - val_loss: 1.3961


In [17]:
image_path = "/kaggle/input/datasets/anonngc/fer2013/test/happy/PrivateTest_10077120.jpg"

In [20]:
test_ds

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [23]:
predict= model.predict(test_ds)

113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 120ms/step


In [24]:
predict

array([[9.1680484e-03, 1.7075853e-04, 7.0966189e-03, ..., 5.1170508e-03,
        7.6382728e-03, 1.0879283e-02],
       [3.3867025e-01, 1.2117871e-03, 3.6187875e-01, ..., 2.1540554e-02,
        8.9139543e-02, 1.2568501e-01],
       [3.0576599e-01, 5.0291792e-03, 1.8779664e-01, ..., 1.0480285e-01,
        3.6872172e-01, 1.1980272e-02],
       ...,
       [6.3753918e-02, 7.8546850e-04, 3.8561597e-02, ..., 1.3429050e-01,
        1.1162249e-01, 1.6720219e-02],
       [3.9172929e-02, 4.7890102e-03, 1.1210953e-02, ..., 5.4223090e-01,
        3.4093861e-02, 9.6087102e-03],
       [7.0844270e-02, 1.0321965e-02, 3.0506087e-02, ..., 2.5855467e-02,
        1.6290775e-02, 1.4178493e-02]], dtype=float32)

In [31]:
allpred,alltrue =[],[]

for images,labels in test_ds:
    pred_prob = model.predict(images)
    pred = pred_prob.argmax(1)

    allpred.extend(pred)
    alltrue.extend(labels.numpy())

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
2/2 ━━━━━━━━

In [32]:
alltrue[:5]

[np.int32(4), np.int32(3), np.int32(3), np.int32(5), np.int32(6)]

In [33]:
allpred[:5]

[np.int64(4), np.int64(3), np.int64(3), np.int64(4), np.int64(6)]